# Train the temporal model (stacked RNN)

Set `CONDITION` below and run the whole notebook. Run it once with `"po"` and
once with `"raw"`; each run writes `results/rnn_<condition>.json`. This json is used to generate final overservations.

In [ ]:
CONDITION = "po" # "po" = participant-only clips, "raw" = full interview recordings

import common
from common import (SEGMENT_LENGTHS, SEEDS, load_metadata, load_features, run_rnn_seeds, save_results, device)

print("Condition:", CONDITION)
print("Features :", common.FEATURE_PATHS[CONDITION])
print("Device   :", device)
print("Seeds    :", SEEDS)

Condition: po
Features : ./androids_is09_participant_clips.npz
Device   : cuda
Seeds    : (0, 1, 2, 3, 4)


In [7]:
interview_df, label_of, gender_of, interview_folds = load_metadata()
data = load_features(CONDITION)

print(f"Speakers: {len(data)} | labelled: {len(label_of)}")
print("Folds:", {k: len(v) for k, v in interview_folds.items()})

Speakers: 116 | labelled: 116
Folds: {1: 24, 2: 23, 3: 23, 4: 23, 5: 23}


## Training

For every segment length the cross-validation is repeated `R = 5` times with
different random initialisations, and the repetitions are averaged **within
each fold**. The stored metric lists therefore hold one entry per fold.

In [8]:
results = {}

for seg_len in SEGMENT_LENGTHS:
    print(f"\n=== segment length {seg_len} ===")

    smv_folds, wa_folds, loss_curves, pooled = run_rnn_seeds(
        data, label_of, interview_folds, seg_len,
        seeds=SEEDS, hidden_size=70, epochs=30
    )

    results[str(seg_len)] = {
        "smv": smv_folds,
        "wa": wa_folds,
        "pooled": pooled,
        "loss_curves": {str(k): v for k, v in loss_curves.items()},   # seed 0 only
    }

    import numpy as np
    for name, folds in [("SMV", smv_folds), ("WA", wa_folds)]:
        f1 = np.array([f["f1"] for f in folds])
        acc = np.array([f["acc"] for f in folds])
        print(f"  {name}: acc {acc.mean()*100:.1f} ± {acc.std(ddof=1)*100:.1f}   "
              f"F1 {f1.mean()*100:.1f} ± {f1.std(ddof=1)*100:.1f}")


=== segment length 32 ===
  SMV: acc 71.7 ± 5.0   F1 68.0 ± 9.8
  WA: acc 73.5 ± 3.5   F1 71.4 ± 7.0

=== segment length 64 ===
  SMV: acc 69.3 ± 3.0   F1 65.6 ± 7.0
  WA: acc 71.7 ± 1.7   F1 70.0 ± 8.0

=== segment length 128 ===
  SMV: acc 70.7 ± 4.1   F1 68.1 ± 10.8
  WA: acc 71.1 ± 3.5   F1 69.5 ± 9.6

=== segment length 256 ===
  SMV: acc 64.9 ± 6.3   F1 64.2 ± 10.3
  WA: acc 66.8 ± 6.8   F1 66.7 ± 10.9

=== segment length 512 ===
  SMV: acc 60.2 ± 7.6   F1 63.2 ± 7.5
  WA: acc 61.8 ± 6.1   F1 65.1 ± 5.8

=== segment length 1024 ===
  SMV: acc 57.6 ± 11.3   F1 59.5 ± 14.8
  WA: acc 59.0 ± 11.0   F1 61.7 ± 13.9


In [9]:
save_results({
    "condition": CONDITION,
    "model": "rnn",
    "seeds": list(SEEDS),
    "segment_lengths": SEGMENT_LENGTHS,
    "hidden_size": 70,
    "num_layers": 2,
    "epochs": 30,
    "results": results,
}, f"rnn_{CONDITION}")

Saved results\rnn_po.json


'results\\rnn_po.json'